In [1]:
!nvidia-smi


Sun Mar  1 16:01:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import numpy as np
from numba import cuda

@cuda.jit
def bitonic_sort_step(values, j, k):
    i = cuda.grid(1)

    if i < values.size:
        ixj = i ^ j

        if ixj > i:
            # Ascending order
            if (i & k) == 0:
                if values[i] > values[ixj]:
                    temp = values[i]
                    values[i] = values[ixj]
                    values[ixj] = temp
            # Descending order
            else:
                if values[i] < values[ixj]:
                    temp = values[i]
                    values[i] = values[ixj]
                    values[ixj] = temp


def bitonic_sort_gpu(arr):
    n = arr.size

    d_arr = cuda.to_device(arr)

    threads_per_block = 256
    blocks = (n + threads_per_block - 1) // threads_per_block

    k = 2
    while k <= n:
        j = k >> 1
        while j > 0:
            bitonic_sort_step[blocks, threads_per_block](d_arr, j, k)
            cuda.synchronize()
            j >>= 1
        k <<= 1

    return d_arr.copy_to_host()


if __name__ == "__main__":
    arr = np.array([3, 7, 4, 8, 6, 2, 1, 5], dtype=np.int32)

    print("Original array:")
    print(arr)

    sorted_arr = bitonic_sort_gpu(arr)

    print("Sorted array:")
    print(sorted_arr)

Original array:
[3 7 4 8 6 2 1 5]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Sorted array:
[1 2 3 4 5 6 7 8]
